In [ ]:
!nvidia-smi

Tue Nov 24 03:24:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install torch torchvision onnx onnxscript onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 114.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 28.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 127.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [onnxscript]


In [ ]:
%%writefile model.py

import torch
import torch.nn as nn

class SimpleModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):

        x = x.view(x.size(0), 784)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x

print("Model architecture created successfully!")

Writing model.py


In [ ]:
!ls

model.py  sample_data


In [ ]:

%%writefile train.py

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from model import SimpleModel

transform = transforms.ToTensor()

train_data = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

loader = DataLoader(train_data, batch_size=64, shuffle=True)

model = SimpleModel()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(2):
    for images, labels in loader:
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch:", epoch + 1, "Loss:", loss.item())

torch.save(model.state_dict(), "model.pth")

print("Trained checkpoint saved!")


Writing train.py


In [ ]:
!python train.py

Model architecture created successfully!
100% 9.91M/9.91M [00:00<00:00, 11.9MB/s]
100% 28.9k/28.9k [00:00<00:00, 400kB/s]
100% 1.65M/1.65M [00:00<00:00, 3.75MB/s]
100% 4.54k/4.54k [00:00<00:00, 14.9MB/s]
Epoch: 1 Loss: 0.17401258647441864
Epoch: 2 Loss: 0.10150070488452911
Trained checkpoint saved!


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'model.py', '__pycache__', 'train.py', 'model.pth', 'data', 'sample_data']


In [ ]:
%%writefile export_onnx.py

import torch
from model import SimpleModel

model = SimpleModel()

model.load_state_dict(
    torch.load("model.pth", map_location="cpu")
)

model.eval()

dummy_input = torch.randn(1, 1, 28, 28)

torch.onnx.export(
    model,
    (dummy_input,),
    "model.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamo=True
)

print("ONNX model exported successfully!")

Writing export_onnx.py


In [ ]:
!python export_onnx.py

Model architecture created successfully!
[torch.onnx] Obtain model graph for `SimpleModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX model exported successfully!


In [ ]:
import onnx

model = onnx.load("model.onnx")

onnx.checker.check_model(model)

print("ONNX model is valid!")

ONNX model is valid!


In [ ]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession("model.onnx")

input_data = np.random.randn(1, 1, 28, 28).astype(np.float32)

result = session.run(None, {"input": input_data})

print("ONNX inference successful!")
print("Output shape:", result[0].shape)

ONNX inference successful!
Output shape: (1, 10)


In [ ]:
!nvidia-smi

Mon Nov 23 06:58:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!which trtexec

In [ ]:
!nvidia-smi

Mon Nov 23 07:04:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import files

files.download("model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("model.onnx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

print("MODEL ARTIFACT LIFECYCLE")

for file in ["model.py", "train.py", "model.pth", "export_onnx.py", "model.onnx"]:
    if os.path.exists(file):
        print(file, "- Successfully created")
    else:
        print(file, "- Missing")

MODEL ARTIFACT LIFECYCLE
model.py - Missing
train.py - Missing
model.pth - Missing
export_onnx.py - Missing
model.onnx - Missing
